# FactPy v0.1 Capabilities — End-to-End Demo

Walks through the five shipped v0.1 capability lines on a single fixture:

1. **Check** — verify a derivation binding holds against committed facts.
2. **Diagnose** — locate the failed atom for a binding that does *not* hold.
3. **Fact Overlay Check** — what-if a fact were different, without writing the ledger.
4. **Why-not Universe Diagnose** — green/red board over an explicit candidate universe, with per-row Diagnose mapping.
5. **Evaluator Frontier Trace** — native frontier rows for failed where-body branches.

This notebook is **assertion-bearing**: each phase asserts the expected outcome. Run cells top-to-bottom; an assertion failure means a real integration drift between capabilities. The companion `examples/11_capabilities_e2e_demo.py` is the deterministic smoke target tracked by `tests/test_examples_capabilities_demo.py`.

## Setup

Imports + repo `src/` path injection so the notebook works whether you launch it from the repo root or from `examples/`.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import sys
from typing import Any

_repo_root = Path.cwd()
if not (_repo_root / 'src').exists() and (_repo_root.parent / 'src').exists():
    _repo_root = _repo_root.parent
_src_dir = _repo_root / 'src'
if str(_src_dir) not in sys.path:
    sys.path.insert(0, str(_src_dir))

from kernel.application import (
    build_schema_index,
    check_derivation_binding,
    check_fact_overlay_binding,
    check_why_not_universe,
    diagnose_derivation_binding,
    entity_info,
    field_predicate,
    resolve_selector,
)
from kernel.application.protocol import (
    CheckRequest,
    CompiledDerivationPlan,
    CompiledHeadCall,
    DiagnoseRequest,
    EntitySelector,
    FactOverlayCheckRequest,
    FactValueOverride,
    WhyNotUniverseRequest,
)
from kernel.core.evidence.write_protocol import set_field
from kernel.core.rules.frontier import evaluate_native_where_frontier
from kernel.core.store import Store
from kernel.sdk import Entity, Field, Identity, compile_schema_from_classes


## Schema and fixture

Model a single `Person` entity with `name` (identity), `age`, and `region`. Seed three rows: **alice** (25, us), **bob** (30, eu), **carol** (28, us).

Build one derivation plan whose body is `person.exists($p) ∧ age($p, $age) ∧ region($p, $region)` and whose head exposes `($p, $age, $region)`.

In [2]:
class Person(Entity):
    name: str = Identity(primary_key=True)
    age: int = Field(cardinality='single')
    region: str = Field(cardinality='single')


@dataclass(frozen=True)
class SeededPerson:
    e_ref: str
    age: int
    region: str
    exists_asrt_id: str
    name_asrt_id: str
    age_asrt_id: str
    region_asrt_id: str
    exists_pred_id: str
    name_pred_id: str
    age_pred_id: str
    region_pred_id: str


def _binding(*items):
    return tuple(sorted(items, key=lambda item: item[0]))


def _seed_person(store, index, *, name, age, region):
    ref = resolve_selector(
        EntitySelector(entity_type='Person', identity={'name': name}),
        index=index,
    )
    info = entity_info(index, 'Person')
    encoded = ref.encoded_ref or ''
    exists_asrt_id = set_field(store.ledger, info.exists_predicate_id, encoded, [])
    name_pred_id = info.identity_predicates['name'].pred_id
    name_asrt_id = set_field(store.ledger, name_pred_id, encoded, [('string', name)])
    age_pred_id = field_predicate(index, 'Person', 'age').pred_id
    age_asrt_id = set_field(store.ledger, age_pred_id, encoded, [('int', age)])
    region_pred_id = field_predicate(index, 'Person', 'region').pred_id
    region_asrt_id = set_field(store.ledger, region_pred_id, encoded, [('string', region)])
    return SeededPerson(
        e_ref=encoded,
        age=age,
        region=region,
        exists_asrt_id=exists_asrt_id,
        name_asrt_id=name_asrt_id,
        age_asrt_id=age_asrt_id,
        region_asrt_id=region_asrt_id,
        exists_pred_id=info.exists_predicate_id,
        name_pred_id=name_pred_id,
        age_pred_id=age_pred_id,
        region_pred_id=region_pred_id,
    )


def _ledger_dump(store):
    return '\n'.join(store.ledger._get_connection().iterdump()).encode('utf-8')


schema_ir = compile_schema_from_classes([Person])
store = Store(schema_ir)
index = build_schema_index(schema_ir)
people = {
    'alice': _seed_person(store, index, name='alice', age=25, region='us'),
    'bob':   _seed_person(store, index, name='bob',   age=30, region='eu'),
    'carol': _seed_person(store, index, name='carol', age=28, region='us'),
}

info = entity_info(index, 'Person')
age_pred = field_predicate(index, 'Person', 'age').pred_id
region_pred = field_predicate(index, 'Person', 'region').pred_id
plan = CompiledDerivationPlan(
    derivation_id='capabilities-e2e-person-snapshot',
    version='1.0',
    body_ir=[
        ('pred', info.exists_predicate_id, ['$p']),
        ('pred', age_pred,    ['$p', '$age']),
        ('pred', region_pred, ['$p', '$region']),
    ],
    heads=(CompiledHeadCall(
        target_pred_id=info.exists_predicate_id,
        head_var_names=('$p', '$age', '$region'),
    ),),
)

for name, person in people.items():
    print(f'  seeded {name:5s} -> e_ref={person.e_ref!r}  age={person.age}  region={person.region!r}')


  seeded alice -> e_ref='idref_v1:Person:554dsfkessfad3gk7intaek35mvkvv65n2gkvzhsyd6xyzav53ya'  age=25  region='us'
  seeded bob   -> e_ref='idref_v1:Person:wluh2tdzguto6pzedaxwaogsuimjchkxblghp2g7o5gu4rg5letq'  age=30  region='eu'
  seeded carol -> e_ref='idref_v1:Person:ia2swouom5cewla4pemhak3gjul3l3urn4wmuox5rscr44s423ra'  age=28  region='us'


## Phase 1 — Check

Ask: *does the binding `($p=alice, $age=25, $region=us)` hold against committed facts?* Expected: `passed`, `matched_count=1`, returned binding equals input.

In [3]:
alice = people['alice']
binding_alice_actual = _binding(
    ('$p', alice.e_ref),
    ('$age', 25),
    ('$region', 'us'),
)

result = check_derivation_binding(
    CheckRequest(plan=plan, binding=binding_alice_actual, engine='native'),
    store=store,
)

assert result.status == 'passed', result
assert result.matched_count == 1, result
assert result.matched_binding == binding_alice_actual, result
print(f'Check status        : {result.status}')
print(f'matched_count       : {result.matched_count}')
print(f'matched_binding     : {result.matched_binding}')


Check status        : passed
matched_count       : 1
matched_binding     : (('$age', 25), ('$p', 'idref_v1:Person:554dsfkessfad3gk7intaek35mvkvv65n2gkvzhsyd6xyzav53ya'), ('$region', 'us'))


## Phase 2 — Diagnose

Ask: *binding `($p=alice, $age=99, $region=us)` does not hold — at which atom does it collapse?* Expected: `failed.atom_localized`; the body is ordered `exists ∧ age ∧ region`, alice exists and region matches, so failure is at atom index 1 (age).

In [4]:
binding_alice_wrong_age = _binding(
    ('$p', alice.e_ref),
    ('$age', 99),
    ('$region', 'us'),
)

result = diagnose_derivation_binding(
    DiagnoseRequest(plan=plan, binding=binding_alice_wrong_age, engine='native'),
    store=store,
)

assert result.status == 'failed', result
assert result.failure_kind == 'atom_localized', result
locator = result.diagnostic_payload
assert locator is not None
assert locator.branch_index == 0
assert locator.failed_atom_index == 1
print(f'Diagnose status     : {result.status}')
print(f'failure_kind        : {result.failure_kind}')
print(f'branch_index        : {locator.branch_index}')
print(f'failed_atom_index   : {locator.failed_atom_index}  (0=exists, 1=age, 2=region)')


Diagnose status     : failed
failure_kind        : atom_localized
branch_index        : 0
failed_atom_index   : 1  (0=exists, 1=age, 2=region)


## Phase 3 — Fact Overlay Check

Ask: *if alice's age were 30 instead of 25, would the binding `($p=alice, $age=30, $region=us)` hold?* Expected: a `before/after/diff` summary where `before.failed`, `after.passed`, and `diff.status_changed=True`. **Critical invariant:** the original ledger must be byte-identical after overlay execution.

In [5]:
binding_alice_30 = _binding(
    ('$p', alice.e_ref),
    ('$age', 30),
    ('$region', 'us'),
)
ledger_before = _ledger_dump(store)

result = check_fact_overlay_binding(
    FactOverlayCheckRequest(
        plan=plan,
        binding=binding_alice_30,
        overlay=(
            FactValueOverride(
                asrt_id=alice.age_asrt_id,
                pred_id=alice.age_pred_id,
                e_ref=alice.e_ref,
                old_fact_tuple=(alice.e_ref, 25),
                new_fact_tuple=(alice.e_ref, 30),
                note='demo overlay: Alice turns 30',
            ),
        ),
        engine='native',
    ),
    store=store,
)

assert result.status == 'passed', result
assert result.before is not None and result.before.status == 'failed'
assert result.after  is not None and result.after.status  == 'passed'
assert result.diff   is not None and result.diff.status_changed
assert _ledger_dump(store) == ledger_before, 'Fact Overlay must not write ledger'
print(f'Overlay status      : {result.status}')
print(f'before.status       : {result.before.status}')
print(f'after.status        : {result.after.status}')
print(f'diff.status_changed : {result.diff.status_changed}')
print('ledger byte-identical: True')


Overlay status      : passed
before.status       : failed
after.status        : passed
diff.status_changed : True
ledger byte-identical: True


## Phase 4 — Why-not Universe Diagnose

Ask: *given an explicit universe of three candidate bindings (alice/bob/carol all targeting age=30, us), which pass and which fail — and why for the failures?* Bob actually has `(age=30, region=eu)`, so the universe entry `(bob, 30, eu)` is the only one that matches the actual ledger; alice and carol each fail at the age atom.

In [6]:
alice_30 = _binding(('$p', alice.e_ref), ('$age', 30), ('$region', 'us'))
bob_30   = _binding(('$p', people['bob'].e_ref), ('$age', 30), ('$region', 'eu'))
carol_30 = _binding(('$p', people['carol'].e_ref), ('$age', 30), ('$region', 'us'))

result = check_why_not_universe(
    WhyNotUniverseRequest(
        plan=plan,
        candidate_universe=(alice_30, bob_30, carol_30),
        engine='native',
    ),
    store=store,
)

assert result.status == 'completed', result
assert result.green == (bob_30,)
assert tuple(row.binding for row in result.red) == (alice_30, carol_30)
for row in result.red:
    assert row.diagnostic.status == 'failed'
    assert row.diagnostic.failure_kind == 'atom_localized'
    assert row.diagnostic.diagnostic_granularity == 'atom_localized'
    locator = row.diagnostic.atom_locator
    assert locator is not None and locator.failed_atom_index == 1

print(f'Why-not status      : {result.status}')
print(f'green count         : {len(result.green)} (expected: 1, bob)')
print(f'red count           : {len(result.red)}  (expected: 2, alice + carol)')
for row in result.red:
    name = next(n for n, p in people.items() if p.e_ref == dict(row.binding)['$p'])
    g = row.diagnostic.diagnostic_granularity
    print(f'  red row {name:5s}: granularity={g} atom_index={row.diagnostic.atom_locator.failed_atom_index}')


Why-not status      : completed
green count         : 1 (expected: 1, bob)
red count           : 2  (expected: 2, alice + carol)
  red row alice: granularity=atom_localized atom_index=1
  red row carol: granularity=atom_localized atom_index=1


## Phase 5 — Evaluator Frontier Trace

Ask: *for the where-body `exists($p) ∧ age($p, 99) ∧ region($p, us)` — where does the native evaluator's branch frontier collapse?* Expected: 1 frontier row at `failed_atom_index=1` (the age atom), with `frontier_count=3` (the three persons enter the age atom before all are filtered) and `failure_kind='atom_filter_empty'`. This is the **substrate-layer** view, complementing Why-not's seeded per-row Diagnose with an unseeded structural read.

In [7]:
view_facts = {
    alice.exists_pred_id:  [(p.e_ref,) for p in people.values()],
    alice.age_pred_id:     [(p.e_ref, p.age)    for p in people.values()],
    alice.region_pred_id:  [(p.e_ref, p.region) for p in people.values()],
}
where = [
    ('pred', alice.exists_pred_id, ['$p']),
    ('pred', alice.age_pred_id,    ['$p', 99]),
    ('pred', alice.region_pred_id, ['$p', 'us']),
]

result = evaluate_native_where_frontier(view_facts, where)

assert result.bindings == []
assert len(result.frontier_rows) == 1
row = result.frontier_rows[0]
assert row.branch_index == 0
assert row.failed_atom_index == 1
assert row.atoms_satisfied == 1
assert row.frontier_count == 3
assert row.failure_kind == 'atom_filter_empty'
print(f'Frontier rows       : {len(result.frontier_rows)}')
print(f'  row.branch_index       = {row.branch_index}')
print(f'  row.failed_atom_index  = {row.failed_atom_index}  (1=age atom)')
print(f'  row.frontier_count     = {row.frontier_count}  (3 persons entered the age atom)')
print(f'  row.failure_kind       = {row.failure_kind!r}')


Frontier rows       : 1
  row.branch_index       = 0
  row.failed_atom_index  = 1  (1=age atom)
  row.frontier_count     = 3  (3 persons entered the age atom)
  row.failure_kind       = 'atom_filter_empty'


## Wrap-up

All 5 phases passed. The capabilities compose cleanly on a single fixture:

- **Check** confirms a binding holds.
- **Diagnose** localizes the atom that breaks a failing binding.
- **Fact Overlay Check** answers what-if without writing the ledger.
- **Why-not Universe Diagnose** reports a green/red board with per-row Diagnose mapping (Sibling-with-Diagnose composition).
- **Evaluator Frontier Trace** exposes the unseeded native frontier — the substrate piece that resolves the Why-not Shape B fork.

Capability sources:

| Capability | Module | Anchor |
|---|---|---|
| Check | `kernel.application.derivation_check_runtime` | `docs/blueprints/archive/2026-05-03_check-operation.md` |
| Diagnose | `kernel.application.diagnose_runtime` | `docs/blueprints/archive/2026-05-04_diagnose-operation.md` |
| Fact Overlay Check | `kernel.application.fact_overlay_runtime` | `docs/blueprints/archive/2026-05-04_fact-overlay-capability.md` |
| Why-not Universe Diagnose | `kernel.application.why_not_runtime` | `docs/blueprints/archive/2026-05-05_why-not-universe-diagnose-capability.md` |
| Evaluator Frontier Trace | `kernel.core.rules.frontier` | `docs/blueprints/archive/2026-05-05_evaluator-frontier-trace-capability.md` |
